# GRACE B3-nodiff — Kaggle test (runtime locked)

Chạy full test census, seed 42: Devign 2,732 + ReVeal 2,274. Prompt giữ nguyên target code, graph và cặp demonstration đã được review; bỏ toàn bộ phần `Key difference`/`Security rationale`.

**Cần thiết lập Kaggle:** attach dataset ZIP `grace_b3_nodiff_test_inputs.zip`, attach đúng một model `Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf`, bật Internet và chọn 2x Tesla T4. Sau đó chạy các cell theo thứ tự.

Runtime lock lấy trực tiếp từ `configs/inference_runtime_lock.json`; SHA-256 `bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e`. Notebook dừng nếu model, tokenizer, packages (bao gồm SciPy), GPU/offload hoặc prompt identity không khớp lock. Chỉ đọc test; không tối ưu ngưỡng trên test. Áp dụng nguyên ngưỡng validation đã khóa: Devign 0.92, ReVeal 0.01.

Kaggle Output gồm checkpoint riêng cho từng dataset: `grace_b3_nodiff_devign_test_checkpoint.zip`, `grace_b3_nodiff_reveal_test_checkpoint.zip`, cùng `grace_diagnostics.zip`.

Test prompts dùng code và graph của từng mẫu test; demo là cặp vulnerable/patched nguồn train đã được Dung và Hiếu duyệt, giữ nguyên như validation. Label test chỉ nằm trong metadata, không nằm trong prompt.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

SESSION_STARTED = time.time()
DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    target = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Diagnostic archive:', target, flush=True)
    return target

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Start {name}; log={log}')
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        is_inference = name.startswith('inference_')
        import queue, threading
        q=queue.Queue()
        def drain():
            for item in proc.stdout: q.put(item)
            q.put(None)
        threading.Thread(target=drain,daemon=True).start()
        status_arg=Path(cmd[cmd.index('--status')+1]) if is_inference and '--status' in cmd else None
        while True:
            try: line=q.get(timeout=60 if is_inference else 1)
            except queue.Empty:
                snapshot={}
                try: snapshot=json.loads(status_arg.read_text(encoding='utf-8')) if status_arg and status_arg.is_file() else {}
                except Exception as exc: snapshot={'status_read_error':repr(exc)}
                runs=snapshot.get('runs',{})
                run_state=next(iter(runs.values()),{}) if runs else {}
                output_path=Path(run_state.get('output',''))/'records.jsonl' if run_state.get('output') else None
                records_n=sum(1 for _ in output_path.open(encoding='utf-8')) if output_path and output_path.is_file() else 0
                total=run_state.get('expected_records') or run_state.get('count') or '?'
                note(f'HEARTBEAT {name}: pid={proc.pid}, state={snapshot.get("state","starting")}, records={records_n}/{total}, status={status_arg}')
                continue
            if line is None: break
            f.write(line); f.flush()
            if line.strip(): print(line,end='',flush=True)
        code = proc.wait()
    note(f'{name} exit={code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    names=('llama-cpp-python','numpy','huggingface-hub','tokenizers','transformers','scikit-learn','scipy')
    out={}
    for name in names:
        try: out[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: out[name]='not installed'
    note('Packages: '+json.dumps(out,sort_keys=True))
    return out

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace=''.join(traceback.format_exception(etype,evalue,tb))
    try:
        (DIAG/'exceptions.log').open('a',encoding='utf-8').write(f'\n{datetime.now(timezone.utc).isoformat()}\n{trace}\n')
        note(f'ERROR {etype.__name__}: {evalue}')
        versions(); diagnostics_zip()
    except Exception as exc:
        print('Could not write diagnostics:',exc,flush=True)
    shell.showtraceback((etype,evalue,tb),tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,),capture_failure)
note('Python: '+sys.version)
note('Platform: '+platform.platform())
for loc in ('/kaggle/working','/kaggle/input'):
    try:
        u=shutil.disk_usage(loc); note(f'Disk {loc}: free={u.free} total={u.total}')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name,cmd in [('gpu',['nvidia-smi']),('cuda_compiler',['nvcc','--version']),('cmake',['cmake','--version']),('pip',[sys.executable,'-m','pip','--version'])]:
    try: run_logged(cmd,name)
    except (OSError,subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()

In [ ]:
import hashlib
from pathlib import PurePosixPath

ARM='B3-nodiff'
SEED=42
SPLIT='test'
BUNDLE_NAME='grace_b3_nodiff_test_inputs.zip'
BUNDLE_SHA256='c35cbd864a0c3a912ac7a4e22eb4b546c56f8e34c4222e02d191011aeca1d433'
RUNTIME_LOCK_SHA256='bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e'
KAGGLE_INPUT_ROOT=Path('/kaggle/input')
WORK=Path('/kaggle/working/b3_nodiff_package')
WORK.mkdir(parents=True,exist_ok=True)
expanded_bundle_members=set()

def sha_bytes(raw): return hashlib.sha256(raw).hexdigest()
def verify_seal(obj,key='seal_sha256'):
    unsigned={k:v for k,v in obj.items() if k!=key}
    actual=sha_bytes(json.dumps(unsigned,sort_keys=True,ensure_ascii=False,indent=2).encode('utf-8'))
    if actual!=obj.get(key): raise ValueError(f'Invalid {key}: expected {obj.get(key)}, got {actual}')

def find_input_files(name):
    found=[]
    for folder,dirs,files in os.walk(KAGGLE_INPUT_ROOT,followlinks=True):
        if name in files: found.append(Path(folder)/name)
    return sorted({str(p.resolve()):p for p in found}.values(),key=lambda p:str(p))

archives=find_input_files(BUNDLE_NAME)
payloads={}
if archives:
    if len(archives)!=1: raise FileNotFoundError(f'Expected at most one {BUNDLE_NAME}, found {archives}')
    archive_path=archives[0]
    if sha_bytes(archive_path.read_bytes())!=BUNDLE_SHA256: raise ValueError('Input bundle SHA-256 mismatch')
    note(f'Input archive verified: {archive_path} sha256={BUNDLE_SHA256}')
    with zipfile.ZipFile(archive_path) as z:
        names=z.namelist()
        if len(names)!=len(set(names)): raise ValueError('Duplicate archive member')
        for info in z.infolist():
            if info.is_dir(): continue
            name=info.filename
            if '\\' in name or ':' in name or PurePosixPath(name).is_absolute() or any(x in ('','.','..') for x in name.split('/')):
                raise ValueError(f'Unsafe archive path: {name}')
            if (info.external_attr>>16)&0o170000==0o120000: raise ValueError('Symlink archive member')
            payloads[name]=z.read(info)
else:
    # Kaggle exposes uploaded ZIP contents as extracted files. Walk mounted/symlinked
    # dataset directories explicitly; pathlib.rglob may not descend through every mount.
    manifest_files=find_input_files('delivery_manifest.json')
    manifest_candidates=[]; rejected=[]
    for manifest_file in manifest_files:
        try:
            candidate=json.loads(manifest_file.read_text(encoding='utf-8'))
            verify_seal(candidate)
            identity=(candidate.get('arm'),candidate.get('split'),candidate.get('seed'),candidate.get('runtime_contract_sha256'))
            wanted=(ARM,SPLIT,SEED,RUNTIME_LOCK_SHA256)
            if identity!=wanted:
                rejected.append(f'{manifest_file}: scope/runtime identity {identity!r} != {wanted!r}')
                continue
            members=candidate.get('files_sha256')
            if not isinstance(members,dict) or not members:
                rejected.append(f'{manifest_file}: files_sha256 missing/empty')
                continue
            missing=[]; missing_bundles=[]; bad=[]
            for rel,digest in members.items():
                relpath=PurePosixPath(rel)
                if relpath.is_absolute() or any(part in ('','.','..') for part in rel.split('/')) or '\\\\' in rel:
                    bad.append(f'unsafe member path {rel!r}'); continue
                member=manifest_file.parent.joinpath(*relpath.parts)
                if not member.is_file():
                    if len(relpath.parts)==2 and relpath.parts[0] in ('devign','reveal') and relpath.suffix=='.zip': missing_bundles.append(rel)
                    else: missing.append(rel)
                elif sha_bytes(member.read_bytes())!=digest: bad.append(f'hash mismatch {rel}')
            if missing or bad:
                rejected.append(f'{manifest_file}: missing={missing[:8]}; invalid={bad[:8]}')
                continue
            manifest_candidates.append((manifest_file,candidate,missing_bundles))
        except Exception as exc:
            rejected.append(f'{manifest_file}: {type(exc).__name__}: {exc}')
    if len(manifest_candidates)!=1:
        try: top=[str(p) for p in KAGGLE_INPUT_ROOT.iterdir()]
        except Exception as exc: top=[f'<cannot list input root: {exc}>']
        note(f'Input discovery: {len(manifest_files)} delivery_manifest.json file(s); {len(manifest_candidates)} verified; top-level={top}; rejected={rejected[:12]}')
        raise FileNotFoundError(f'Expected exactly one verified extracted B3-nodiff delivery under {KAGGLE_INPUT_ROOT}; found {len(manifest_candidates)}. See Input discovery log above for paths and rejection reasons.')
    manifest_file,delivery,expanded_bundle_members=manifest_candidates[0]
    expanded_bundle_members=set(expanded_bundle_members)
    payloads={'delivery_manifest.json':manifest_file.read_bytes()}
    for rel in delivery['files_sha256']:
        member=manifest_file.parent/PurePosixPath(rel)
        if member.is_file(): payloads[rel]=member.read_bytes()
    note(f'Kaggle-extracted input verified from {manifest_file.parent}: sealed manifest; {len(delivery["files_sha256"])-len(expanded_bundle_members)} file hashes verified; nested archives auto-expanded={sorted(expanded_bundle_members)}')
if 'delivery_manifest.json' not in payloads: raise ValueError('Missing delivery manifest')
delivery=json.loads(payloads['delivery_manifest.json'].decode('utf-8'))
verify_seal(delivery)
if delivery.get('arm')!=ARM or delivery.get('split')!=SPLIT or delivery.get('seed')!=SEED: raise ValueError('Delivery scope mismatch')
if delivery.get('runtime_contract_sha256')!=RUNTIME_LOCK_SHA256: raise ValueError('Delivery points to a different runtime contract')
expected_files=delivery['files_sha256']
provided=set(payloads)-{'delivery_manifest.json'}; expected=set(expected_files); missing=expected-provided
if provided-expected or missing!=expanded_bundle_members: raise ValueError(f'Unexpected/missing delivery members: missing={sorted(missing)}, auto-expanded={sorted(expanded_bundle_members)}, unexpected={sorted(provided-expected)}')
for name,digest in expected_files.items():
    if name in payloads and sha_bytes(payloads[name])!=digest: raise ValueError(f'Member hash mismatch: {name}')
lock_raw=payloads['inference_runtime_lock.json']
if sha_bytes(lock_raw)!=RUNTIME_LOCK_SHA256: raise ValueError('Runtime lock file is not the user-specified lock')
(WORK/'inference_runtime_lock.json').write_bytes(lock_raw)
Path('/kaggle/working/inference_runtime_lock.json').write_bytes(lock_raw)
Path('/kaggle/working/inference_runtime_lock.json').write_bytes(lock_raw)
for name in ('grace_runtime_contract.py','b3_delivery_runtime.py','kaggle_inference.py'):
    raw=payloads[f'support/{name}']
    (Path('/kaggle/working')/name).write_bytes(raw)
EVALUATOR_ROOT=Path('/kaggle/working/grace_diag'); EVALUATOR_ROOT.mkdir(parents=True,exist_ok=True)
for name in ('__init__.py','evaluate_final.py','stats.py','thresholds.py'):
    (EVALUATOR_ROOT/name).write_bytes(payloads[f'shared_evaluator/grace_diag/{name}'])
if '/kaggle/working' not in sys.path: sys.path.insert(0,'/kaggle/working')
from grace_runtime_contract import EXPECTED_CONTRACT_SHA256, load_contract
from b3_delivery_runtime import load_input_root, validate_bundle, checked_records, make_checkpoint, restore_checkpoint, write_zip, seal
if EXPECTED_CONTRACT_SHA256!=RUNTIME_LOCK_SHA256: raise ValueError('Runtime helper has a different lock pin')
RUNTIME=load_contract(WORK/'inference_runtime_lock.json')
THRESHOLD_RAW=payloads['validation_threshold_lock.json']
THRESHOLD_RAW_SHA256='c4c438a413ffafcbcac993be49c12963497a3290a8dc679f2c3e6fee763bdebf'
if sha_bytes(THRESHOLD_RAW)!=THRESHOLD_RAW_SHA256: raise ValueError('Validation threshold lock bytes changed')
THRESHOLD_LOCK=json.loads(THRESHOLD_RAW.decode('utf-8'))
if THRESHOLD_LOCK.get('locked_sha256')!='64dde9b94a44ae56b3abe0b7e6abd6ced2a5320d7fee2785a961c5da225a786c': raise ValueError('Validation threshold lock seal mismatch')
if THRESHOLD_LOCK.get('arm')!=ARM or THRESHOLD_LOCK.get('split')!='validation' or THRESHOLD_LOCK.get('runtime_contract_sha256')!='85d8770839f80260cdc7c27b27b6cce6bf8815675c8593db68fb102589c5cbd6': raise ValueError('Validation threshold source identity mismatch')
if THRESHOLD_LOCK.get('model_sha256')!=RUNTIME['model']['sha256']: raise ValueError('Validation threshold lock model differs from the current runtime lock')

if RUNTIME['model']['sha256']!='f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335': raise ValueError('Locked model digest mismatch')
if RUNTIME['tokenizer']['revision']!='b2cff646eb4bb1d68355c01b18ae02e7cf42d120': raise ValueError('Locked tokenizer revision mismatch')
if RUNTIME['scoring']['name']!='unconstrained_first_token_logits_v1': raise ValueError('Unexpected scoring contract')
if RUNTIME['constructor']['n_ctx']!=8192 or RUNTIME['constructor']['n_batch']!=256 or RUNTIME['constructor']['n_ubatch']!=256: raise ValueError('Constructor differs from lock')

DATASETS={}
for ds in ('devign','reveal'):
    root_rel=f'{ds}/inference_input_manifest.json'
    root_raw=payloads[root_rel]
    root_path=WORK/f'{ds}_inference_input_manifest.json'
    root_path.write_bytes(root_raw)
    root=load_input_root(root_path,sha_bytes(root_raw))
    expected_count=2732 if ds=='devign' else 2274
    if root['scope']!={'arm':ARM,'dataset':ds,'split':SPLIT,'seed':SEED,'count':expected_count}:
        raise ValueError(f'Unexpected {ds} scope: {root["scope"]}')
    for code_name,rel in [('grace_runtime_contract.py','support/grace_runtime_contract.py'),('b3_delivery_runtime.py','support/b3_delivery_runtime.py'),('kaggle_inference.py','support/kaggle_inference.py')]:
        if root['code_sha256'][code_name]!=sha_bytes(payloads[rel]): raise ValueError(f'{ds} runner/helper hash mismatch: {code_name}')
    bundle_rel=f'{ds}/{root["bundle_filename"]}'
    bundle_path=WORK/root['bundle_filename']
    if bundle_rel in payloads:
        bundle_raw=payloads[bundle_rel]
        if sha_bytes(bundle_raw)!=root['bundle_sha256']: raise ValueError(f'{ds} bundle hash mismatch')
        bundle_path.write_bytes(bundle_raw)
        manifest,rows,members=validate_bundle(bundle_path,root)
    elif bundle_rel in expanded_bundle_members:
        dataset_root=manifest_file.parent/ds
        bundle_manifests=[p for p in find_input_files('bundle_manifest.json') if dataset_root in p.parents or p.parent==dataset_root]
        valid_sources=[]
        for bm_path in bundle_manifests:
            candidate_root=bm_path.parent; candidate_members={}; missing_parts=[]; bad_parts=[]
            for rel,digest in root['members_sha256'].items():
                part=candidate_root.joinpath(*PurePosixPath(rel).parts)
                if not part.is_file(): missing_parts.append(rel)
                else:
                    raw=part.read_bytes(); candidate_members[rel]=raw
                    if sha_bytes(raw)!=digest: bad_parts.append(rel)
            if not missing_parts and not bad_parts: valid_sources.append((candidate_root,candidate_members))
        if len(valid_sources)!=1:
            raise FileNotFoundError(f'{ds}: expected one extracted inner bundle matching its member hashes; candidates={[str(x) for x in bundle_manifests]}, valid={[str(x[0]) for x in valid_sources]}')
        source_root,members=valid_sources[0]
        write_zip(bundle_path,members)
        reconstructed_sha=sha_bytes(bundle_path.read_bytes())
        original_sha=root['bundle_sha256']
        root['bundle_sha256']=reconstructed_sha
        root['bundle_transport']={'type':'kaggle_nested_zip_auto_extracted','declared_source_bundle_sha256':original_sha,'reconstructed_bundle_sha256':reconstructed_sha,'verified_by':'sealed bundle manifest and every members_sha256 entry'}
        root['seal_sha256']=seal(root)
        derived_raw=(json.dumps(root,ensure_ascii=False,indent=2)+'\n').encode('utf-8')
        root_path.write_bytes(derived_raw)
        root_sha256=sha_bytes(derived_raw)
        root=load_input_root(root_path,root_sha256)
        manifest,rows,members=validate_bundle(bundle_path,root)
        note(f'{ds}: Kaggle auto-expanded nested ZIP at {source_root}; reconstructed transport ZIP from verified members; declared source SHA={original_sha}')
    else:
        raise FileNotFoundError(f'{ds}: packed or auto-expanded prompt bundle is missing: {bundle_rel}')
    if root.get('threshold_lock_sha256')!=THRESHOLD_RAW_SHA256 or root.get('threshold')!=THRESHOLD_LOCK['entries'][ds]['tau_star']: raise ValueError(f'{ds} validation threshold lock mismatch')
    if len(rows)!=root['scope']['count']: raise ValueError(f'{ds} census count mismatch')
    if any('Key difference:' in r['prompt'] or 'Security rationale:' in r['prompt'] for r in rows): raise ValueError(f'{ds} prompt still has diff explanation')
    DATASETS[ds]={'root':root,'root_path':root_path,'root_sha256':sha_bytes(root_path.read_bytes()),'bundle':bundle_path,'manifest':manifest,'rows':rows}
    note(f'{ds} bundle verified: {len(rows)} prompts, pair={manifest["source_pair_id"]}, prompt_sha={manifest["prompts_sha256"]}')
print('Verified test rows:',{k:len(v['rows']) for k,v in DATASETS.items()})
print('Runtime lock:',RUNTIME_LOCK_SHA256)

In [ ]:
import urllib.request
build=RUNTIME['build']
os.environ['CMAKE_ARGS']=(f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}"
    f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}"
    f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}")
for key in ('CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE'): os.environ[key]=build[key]
note('Locked build environment: '+json.dumps({k:os.environ[k] for k in ('CMAKE_ARGS','CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE')},sort_keys=True))
packages=[f'{name}=={version}' for name,version in RUNTIME['packages'].items() if name!=RUNTIME['backend_source']['name']]
packages+=['typing-extensions>=4.5.0','diskcache>=5.6.1','jinja2>=2.11.3']
run_logged([sys.executable,'-m','pip','install','--no-cache-dir',*packages],'install_dependencies')
source=RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json",timeout=60) as response: metadata=json.load(response)
artifact=next(x for x in metadata['urls'] if x['filename']==source['sdist_filename'])
if artifact['digests']['sha256']!=source['sdist_sha256']: raise ValueError('Backend source registry SHA-256 mismatch')
archive_path=Path('/kaggle/working')/source['sdist_filename']
with urllib.request.urlopen(artifact['url'],timeout=120) as response,archive_path.open('wb') as target: shutil.copyfileobj(response,target)
if hashlib.sha256(archive_path.read_bytes()).hexdigest()!=source['sdist_sha256']: raise ValueError('Backend source archive SHA-256 mismatch')
run_logged([sys.executable,'-m','pip','install','-v','--no-cache-dir','--no-deps','--force-reinstall',str(archive_path)],'install_llama_cuda')
from grace_runtime_contract import assert_runtime
locked_versions,gpu_inventory=assert_runtime(RUNTIME)
note('Exact locked package versions: '+json.dumps(locked_versions,sort_keys=True)); note('GPU inventory: '+gpu_inventory)
(DIAG/'inference_runtime_lock.json').write_bytes((WORK/'inference_runtime_lock.json').read_bytes())
(DIAG/'locked_package_versions.json').write_text(json.dumps(locked_versions,indent=2,sort_keys=True),encoding='utf-8')
freeze=subprocess.run([sys.executable,'-m','pip','freeze','--all'],check=True,capture_output=True,text=True)
(DIAG/'pip_freeze_all.txt').write_text(freeze.stdout,encoding='utf-8')
diagnostics_zip()

In [ ]:
from grace_runtime_contract import assert_runtime, digest
assert_runtime(RUNTIME)
import llama_cpp
versions()
note('llama-cpp-python GPU offload support: '+str(llama_cpp.llama_supports_gpu_offload()))
if not llama_cpp.llama_supports_gpu_offload(): raise RuntimeError('CUDA backend unavailable under locked runtime')
gpu_files=find_input_files(RUNTIME['model']['filename'])
if len(gpu_files)!=1: raise FileNotFoundError(f'Need exactly one locked GGUF model; found {gpu_files}')
MODEL_PATH=gpu_files[0]
if digest(MODEL_PATH)!=RUNTIME['model']['sha256']:
    raise ValueError('Model file SHA-256 differs from runtime lock')
note(f'Model input pinned: {MODEL_PATH} sha256={RUNTIME["model"]["sha256"]}')
print('Hardware/runtime preflight complete. Worker requires and verifies full 49-layer T4 offload before scoring.')
diagnostics_zip()
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(RUNTIME['tokenizer']['repo'],revision=RUNTIME['tokenizer']['revision'])
for ds in ('devign','reveal'):
    rows=DATASETS[ds]['rows']; budget=DATASETS[ds]['manifest']['prompt_budget_tokens']
    for index,item in enumerate(rows,1):
        encoded=tokenizer.apply_chat_template([{'role':'user','content':item['prompt']}],tokenize=True,add_generation_prompt=True,return_dict=False)
        if len(encoded)>budget: raise ValueError(f'{ds}/{item["id"]}: prompt uses {len(encoded)} tokens, exceeds locked context {budget}')
        if index%250==0 or index==len(rows): note(f'Token budget preflight {ds}: {index}/{len(rows)} checked; all <= {budget}')


In [ ]:
MAX_RUNTIME_HOURS=10.5
CHECKPOINT_RESERVE_SECONDS=300
OUT_ROOT=Path('/kaggle/working/b3_nodiff_test')
CHECKPOINT_PATHS={'devign':None,'reveal':None}  # Set an explicit Kaggle checkpoint input path only when resuming.
OUT_ROOT.mkdir(parents=True,exist_ok=True)
deadline=SESSION_STARTED+MAX_RUNTIME_HOURS*3600
summary={'arm':ARM,'seed':SEED,'split':SPLIT,'runtime_contract_sha256':RUNTIME_LOCK_SHA256,
         'threshold_lock_sha256':THRESHOLD_RAW_SHA256,'thresholds':{},'datasets':{},'state':'running'}
for ds in ('devign','reveal'):
    data=DATASETS[ds]; root=data['root']; tau=float(THRESHOLD_LOCK['entries'][ds]['tau_star'])
    if float(root['threshold'])!=tau: raise ValueError(f'{ds}: delivered threshold differs from validation lock')
    summary['thresholds'][ds]={'tau_star':tau,'source':'locked validation_threshold_lock.json','comparison':'score >= threshold'}
    ds_out=OUT_ROOT/ds; status_path=ds_out/f'seed_{SEED}_status.json'
    checkpoint=Path('/kaggle/working')/f'grace_b3_nodiff_{ds}_test_checkpoint.zip'
    try:
        if CHECKPOINT_PATHS[ds] is not None: restore_checkpoint(Path(CHECKPOINT_PATHS[ds]),ds_out,root,data['rows'])
        elif ds_out.exists(): raise ValueError(f'Fresh output directory already exists: {ds_out}; set explicit checkpoint path to resume')
        else: ds_out.mkdir(parents=True)
        done=checked_records(ds_out,root,data['rows'])
        note(f'{ds} test inference start/resume: {done}/{len(data["rows"])} complete; locked tau={tau:.2f}')
        if done<len(data['rows']) and time.time()<deadline-CHECKPOINT_RESERVE_SECONDS:
            (Path('/kaggle/working')/'inference_input_manifest.json').write_bytes(data['root_path'].read_bytes())
            env=os.environ.copy(); env['B3_INPUT_ROOT_SHA256']=data['root_sha256']
            env['PYTHONPATH']='/kaggle/working'+os.pathsep+env.get('PYTHONPATH','')
            command=[sys.executable,'-u','/kaggle/working/kaggle_inference.py','--model',str(MODEL_PATH),
                '--model-revision','local-input','--seed',str(SEED),'--arm',ARM,
                '--expected-model-sha256',RUNTIME['model']['sha256'],'--gpu-layers',str(RUNTIME['constructor']['n_gpu_layers']),
                '--deadline-epoch',str(deadline),'--checkpoint-reserve-seconds',str(CHECKPOINT_RESERVE_SECONDS),
                '--status',str(status_path),'--run',ds,SPLIT,str(data['bundle']),str(ds_out/'seed_42'/ds/SPLIT)]
            run_logged(command,f'inference_{ds}_seed_{SEED}',env=env)
        done=checked_records(ds_out,root,data['rows'])
        make_checkpoint(ds_out,checkpoint,root,data['rows'])
        summary['datasets'][ds]={'expected':len(data['rows']),'completed':done,'checkpoint':str(checkpoint),
            'tau_star':tau,'state':'complete' if done==len(data['rows']) else 'checkpointed_or_incomplete'}
        note(f'{ds} test progress complete: {done}/{len(data["rows"])}; locked tau={tau:.2f}; checkpoint={checkpoint}')
    except Exception as exc:
        summary['datasets'][ds]={'state':'error','error':repr(exc),'status_path':str(status_path)}
        note(f'{ds} test run error: {exc}')
        raise
summary['state']='complete' if all(summary['datasets'].get(ds,{}).get('completed')==len(DATASETS[ds]['rows']) for ds in ('devign','reveal')) else 'checkpointed_or_incomplete'
if summary['state']=='complete':
    # Put records in the shared evaluator's canonical runs/final/<dataset>/<split>/<arm>/<seed> layout.
    EVAL_RUNS=OUT_ROOT/'runs'/'final'
    for ds in ('devign','reveal'):
        src_folder=OUT_ROOT/ds/f'seed_{SEED}'/ds/SPLIT
        dst_folder=EVAL_RUNS/ds/SPLIT/ARM/str(SEED); dst_folder.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src_folder/'records.jsonl',dst_folder/'records.jsonl')
        shutil.copy2(src_folder/'run_manifest.json',dst_folder/'run_manifest.json')
    # Adapter is local to this run; frozen configs/thresholds.json is never altered.
    locked_entries={f'{ds}|{ARM}|{SEED}':{'dataset':ds,'arm':ARM,'seed':SEED,'tau':float(THRESHOLD_LOCK['entries'][ds]['tau_star']),
        'policy':'locked_validation','source_threshold_lock_sha256':THRESHOLD_LOCK['locked_sha256']} for ds in ('devign','reveal')}
    lock_body={'schema':'grace_diag_threshold_adapter_v1','source_lock_sha256':THRESHOLD_LOCK['locked_sha256'],
        'source_runtime_contract_sha256':THRESHOLD_LOCK['runtime_contract_sha256'],'thresholds':locked_entries}
    lock_body['locked_sha256']=hashlib.sha256(json.dumps({k:v for k,v in lock_body.items() if k!='locked_sha256'},
        indent=2,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
    evaluator_thresholds=OUT_ROOT/'b3_nodiff_thresholds_for_shared_evaluator.json'
    evaluator_thresholds.write_text(json.dumps(lock_body,indent=2,ensure_ascii=False,sort_keys=True)+'\n',encoding='utf-8')
    from grace_diag.evaluate_final import load_runs, table_a
    from grace_diag.thresholds import load_locked
    shared_locked=load_locked(evaluator_thresholds)
    evaluator_metrics=OUT_ROOT/'shared_evaluator_output'; evaluator_metrics.mkdir(parents=True,exist_ok=True)
    evaluator_data=load_runs(EVAL_RUNS)
    used=table_a(evaluator_data,evaluator_metrics,shared_locked)
    if len(used)!=2 or any(used.get((ds,ARM,SEED))!=float(THRESHOLD_LOCK['entries'][ds]['tau_star']) for ds in ('devign','reveal')):
        raise ValueError(f'Shared evaluator did not apply both locked thresholds: {used}')
    (evaluator_metrics/'thresholds_used.json').write_text(json.dumps({'source_validation_threshold_lock_sha256':THRESHOLD_LOCK['locked_sha256'],
        'source_runtime_contract_sha256':THRESHOLD_LOCK['runtime_contract_sha256'],
        'used':{f'{k[0]}|{k[1]}|{k[2]}':v for k,v in used.items()}},indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    summary['shared_evaluator']={'name':'grace_diag.evaluate_final.table_a','thresholds_file':str(evaluator_thresholds),
        'metrics_csv':str(evaluator_metrics/'table_A_metrics.csv'),
        'thresholds_used':{f'{k[0]}|{k[1]}|{k[2]}':v for k,v in used.items()}}
    note(f'Shared evaluator complete: {evaluator_metrics}; thresholds={used}')
else:
    summary['shared_evaluator']={'state':'skipped_until_full_census'}
(OUT_ROOT/'batch_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
note('Test run summary: '+json.dumps(summary,ensure_ascii=False))
diagnostics_zip()


## Kaggle run notes

This run is validation-only. It writes separate resumable checkpoints for Devign and ReVeal. Do not interpret validation scores as test results. The worker refuses to silently truncate prompts, change tokenizer IDs, append to outputs with a different contract, or continue if GPU offload evidence is incomplete.